# Runnable Interface & Batch in LangChain

## What is Runnable?
Every component in LangChain (LLMs, prompts, parsers, chains) implements the **Runnable** interface.
This gives them a **common set of methods** so they can be composed and used uniformly.

## Core Methods

| Method | Description |
|---|---|
| `.invoke(input)` | Single input → single output |
| `.batch([...])` | Multiple inputs → multiple outputs (parallel) |
| `.stream(input)` | Single input → yields tokens as they generate |
| `.ainvoke()` / `.abatch()` / `.astream()` | Async versions of the above |

In [ ]:
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv

load_dotenv(override=True)

llm = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=200)

## 1. `.invoke()` — Single Input

In [ ]:
# invoke: send one message, get one response
response = llm.invoke("What is LangChain in one sentence?")
print(response.content)

## 2. `.batch()` — Multiple Inputs in Parallel

- Sends **all inputs concurrently** (uses threading internally)
- Returns a **list of responses** in the same order as inputs
- Much faster than calling `.invoke()` in a loop

In [ ]:
questions = [
    "What is LangChain?",
    "What is a Runnable in LangChain?",
    "What is Claude AI?",
]

responses = llm.batch(questions)

for q, r in zip(questions, responses):
    print(f"Q: {q}")
    print(f"A: {r.content[:200]}")
    print("-" * 60)

## 3. `.batch()` with `max_concurrency` — Control Parallelism

Limit parallel calls to avoid rate limits.

In [ ]:
responses = llm.batch(
    ["Explain AI", "Explain ML", "Explain DL"],
    config={"max_concurrency": 2}   # only 2 requests run at a time
)

for r in responses:
    print(r.content[:150])
    print("-" * 60)

## 4. `.stream()` — Stream Tokens as They Generate

In [ ]:
# print tokens as they arrive
for chunk in llm.stream("Tell me a fun fact about space."):
    print(chunk.content, end="", flush=True)

## 5. Batch on a Full Chain (Prompt + LLM)

Since both `PromptTemplate` and `ChatAnthropic` are Runnables, chain them with `|` (pipe).
`.batch()` works on the **entire chain**.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence.")
chain = prompt | llm   # both are Runnables

topics = [
    {"topic": "LangChain"},
    {"topic": "Vector Databases"},
    {"topic": "RAG (Retrieval-Augmented Generation)"},
]

results = chain.batch(topics)

for t, r in zip(topics, results):
    print(f"{t["topic"]}: {r.content}")
    print("-" * 60)

## Summary

| Method | Inputs | Output | Use When |
|---|---|---|---|
| `.invoke()` | 1 | 1 | Single query |
| `.batch()` | Many | Many (parallel) | Bulk processing |
| `.stream()` | 1 | Streamed tokens | Real-time UX |

**Key rule:** Any Runnable (LLM, prompt, parser, full chain) supports all these methods.